In [ ]:
#Installing dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-otfscm6t/unsloth_0ea1a3f16f8e4f868dda9a65c16164a6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-otfscm6t/unsloth_0ea1a3f16f8e4f868dda9a65c16164a6
  Resolved https://github.com/unslothai/unsloth.git to commit 3ab282fd402cf6af03660c4d8d89a7d4ba9da1ff
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 23.5 MB/s eta 0:00:0

In [ ]:
#Importing necessary libraries

from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
import json
import pandas as pd

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#Loading Base model

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Will load unsloth/meta-llama-3.1-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
#Adding LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


In [ ]:
#Creating a dataset with 30 examples related software development resume bullet points

train_examples = [
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Built APIs for student platform.",
        "output": "Developed scalable backend APIs for a student platform, improving service reliability and supporting high-volume user workflows."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Worked on frontend for learning app.",
        "output": "Built responsive frontend features for a learning platform, improving usability and streamlining learner-facing workflows."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Used React to make UI components.",
        "output": "Developed reusable React components to improve UI consistency, maintainability, and development velocity across the application."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Made a chatbot using LLMs.",
        "output": "Built an LLM-powered chatbot that delivered context-aware responses, improving answer relevance and user interaction quality."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Used vector database for retrieval.",
        "output": "Integrated vector-based retrieval to support semantic search and improve grounding for LLM-generated responses."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a backend engineering role.",
        "input": "Worked with databases and queries.",
        "output": "Optimized database queries and data access patterns to improve backend performance and support scalable application workflows."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a cloud engineering role.",
        "input": "Deployed app on AWS.",
        "output": "Deployed and maintained cloud-native application services on AWS, improving scalability, availability, and operational efficiency."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a DevOps role.",
        "input": "Set up CI/CD pipelines.",
        "output": "Implemented CI/CD pipelines to automate build, test, and deployment workflows, reducing manual effort and improving release consistency."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a data engineering role.",
        "input": "Processed user activity data.",
        "output": "Built data processing workflows for user activity events, enabling structured analytics and faster insight generation."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Helped improve app performance.",
        "output": "Improved application performance through targeted optimizations, reducing latency and enhancing the overall user experience."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a backend engineering role.",
        "input": "Built microservices for platform.",
        "output": "Engineered backend microservices to support modular platform functionality, improving scalability and service maintainability."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Worked with authentication.",
        "output": "Implemented secure authentication and authorization workflows to protect user access and improve application security."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Fine tuned an LLM.",
        "output": "Fine-tuned an open-source LLM using parameter-efficient training methods to improve domain-specific response quality."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Evaluated model outputs.",
        "output": "Designed evaluation workflows to assess model output quality, identify failure patterns, and guide iterative model improvement."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Built dashboards for analytics.",
        "output": "Developed analytics dashboards to surface key product metrics and support data-informed decision making."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a mobile engineering role.",
        "input": "Worked on Android features.",
        "output": "Implemented Android application features that improved functionality, usability, and consistency across mobile workflows."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a frontend engineering role.",
        "input": "Improved website loading time.",
        "output": "Optimized frontend rendering and asset delivery to reduce page load times and improve user experience."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Used Docker for deployment.",
        "output": "Containerized application services with Docker to standardize environments and simplify deployment workflows."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a cloud engineering role.",
        "input": "Worked with serverless functions.",
        "output": "Developed serverless workflows to support event-driven processing and improve backend scalability."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a backend engineering role.",
        "input": "Created REST APIs using Node.js.",
        "output": "Built RESTful backend services in Node.js to support scalable application workflows and reliable client-server communication."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Built a RAG assistant.",
        "output": "Developed a retrieval-augmented generation assistant that combined semantic retrieval with LLM reasoning to produce context-grounded responses."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Used LangChain in a project.",
        "output": "Built LLM orchestration workflows with LangChain to manage retrieval, prompting, and response generation across AI application features."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Helped with debugging and bug fixing.",
        "output": "Diagnosed and resolved application issues to improve system stability, code quality, and release reliability."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a backend engineering role.",
        "input": "Worked on API integrations.",
        "output": "Integrated third-party APIs and backend services to extend application functionality and improve interoperability."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a data engineering role.",
        "input": "Built ETL pipelines.",
        "output": "Designed ETL pipelines to transform raw data into structured datasets for analytics and downstream processing."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Collaborated with team on features.",
        "output": "Collaborated across engineering teams to deliver production features aligned with product requirements and technical standards."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a frontend engineering role.",
        "input": "Used TypeScript in frontend work.",
        "output": "Built strongly typed frontend features with TypeScript to improve code maintainability and reduce runtime issues."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a DevOps role.",
        "input": "Monitored services in production.",
        "output": "Implemented production monitoring and alerting workflows to improve incident visibility and support faster issue resolution."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Worked with embeddings.",
        "output": "Used embedding-based representations to enable semantic retrieval and improve downstream LLM application performance."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Improved code quality.",
        "output": "Improved code quality through refactoring, testing, and development best practices, increasing maintainability and reducing defects."
    }
]


In [ ]:
#Defining Evaluation examples

eval_examples = [
    {
        "instruction": "Rewrite this resume bullet to be stronger for a software engineering role.",
        "input": "Built backend for app.",
        "output": "Developed backend services for an application, improving scalability, reliability, and support for core user workflows."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Made LLM app for documents.",
        "output": "Built an LLM application for document understanding, enabling context-aware responses from retrieved content."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a cloud engineering role.",
        "input": "Used AWS to host services.",
        "output": "Deployed and managed cloud services on AWS to support scalable, reliable application infrastructure."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a frontend engineering role.",
        "input": "Worked on React pages.",
        "output": "Developed React-based user interfaces that improved consistency, responsiveness, and overall user experience."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a backend engineering role.",
        "input": "Connected app with external APIs.",
        "output": "Integrated external APIs into backend workflows to expand functionality and support reliable data exchange."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a data engineering role.",
        "input": "Built pipelines for analytics.",
        "output": "Developed data pipelines to support structured analytics, reliable data flow, and faster reporting."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for an AI engineering role.",
        "input": "Worked on prompt engineering.",
        "output": "Designed prompt workflows to improve response relevance, consistency, and task performance in LLM-based applications."
    },
    {
        "instruction": "Rewrite this resume bullet to be stronger for a DevOps role.",
        "input": "Automated deployments.",
        "output": "Automated deployment workflows to improve release speed, consistency, and operational reliability."
    }
]

In [ ]:
#Saving these created datasets as JSONL files

with open("train.jsonl", "w", encoding="utf-8") as f:
    for ex in train_examples:
        f.write(json.dumps(ex) + "\n")

with open("eval.jsonl", "w", encoding="utf-8") as f:
    for ex in eval_examples:
        f.write(json.dumps(ex) + "\n")

print("Saved train.jsonl and eval.jsonl")

Saved train.jsonl and eval.jsonl


In [ ]:
#Loading these datasets

dataset=load_dataset("json", data_files={"train": "train.jsonl", "eval": "eval.jsonl"})
dataset



DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 30
    })
    eval: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 8
    })
})

In [ ]:
#Formatting the dataset for training the LLM

EOS_TOKEN = tokenizer.eos_token

def format_resume(example):
    text = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n{example['output']}{EOS_TOKEN}"
    )
    return {"text": text}

dataset["train"] = dataset["train"].map(format_resume)
dataset["eval"] = dataset["eval"].map(format_resume)

In [ ]:
#Example from the formatted dataset

print(dataset["train"][0]["text"])

### Instruction:
Rewrite this resume bullet to be stronger for a software engineering role.

### Input:
Built APIs for student platform.

### Response:
Developed scalable backend APIs for a student platform, improving service reliability and supporting high-volume user workflows.<|end_of_text|>


In [ ]:
#Trainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=120,
        learning_rate=2e-4,
        logging_steps=5,
        eval_steps=10,
        #evaluation_strategy="steps",
        save_strategy="no",
        report_to="none",
        output_dir="outputs",
        seed=42,
    ),
)


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30 | Num Epochs = 30 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss
5,3.573749
10,2.622595
15,1.448503
20,1.067484
25,0.859836
30,0.655330
35,0.534911
40,0.394636
45,0.304483
50,0.273581


TrainOutput(global_step=120, training_loss=0.58891698072354, metrics={'train_runtime': 271.0762, 'train_samples_per_second': 3.541, 'train_steps_per_second': 0.443, 'total_flos': 2076294394920960.0, 'train_loss': 0.58891698072354, 'epoch': 30.0})

In [ ]:
#Saving the fine tuned model

model.save_pretrained("resume_lora_adapter")
tokenizer.save_pretrained("resume_lora_adapter")

('resume_lora_adapter/tokenizer_config.json',
 'resume_lora_adapter/tokenizer.json')

In [ ]:
#Changing to inference mode

FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
#Testing simple examples

prompt = """### Instruction:
Rewrite this resume bullet to be stronger for an AI engineering role.

### Input:
Built chatbot for answering questions.

### Response:
"""

device = next(model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### Instruction:
Rewrite this resume bullet to be stronger for an AI engineering role.

### Input:
Built chatbot for answering questions.

### Response:
Developed an LLM-powered chatbot that delivered context-aware responses, improving answer relevance and user interaction quality.


In [ ]:
#Helper Functions for Evaluation

def make_prompt(example):
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n"
    )

def generate_answer(model, tokenizer, prompt, max_new_tokens=80):
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded.strip()


In [ ]:
#Evaluation on held out set

results = []

for ex in eval_examples:
    prompt = make_prompt(ex)
    prediction = generate_answer(model, tokenizer, prompt)

    results.append({
        "input": ex["input"],
        "expected": ex["output"],
        "prediction": prediction,
    })

df = pd.DataFrame(results)
df

,input,expected,prediction
0,Built backend for app.,"Developed backend services for an application,...",Developed scalable backend services to support...
1,Made LLM app for documents.,Built an LLM application for document understa...,Built an LLM application for semantic processi...
2,Used AWS to host services.,Deployed and managed cloud services on AWS to ...,Deployed serverless workflows on AWS to suppor...
3,Worked on React pages.,Developed React-based user interfaces that imp...,Built responsive React frontend features that ...
4,Connected app with external APIs.,Integrated external APIs into backend workflow...,Integrated third-party APIs and backend servic...
5,Built pipelines for analytics.,Developed data pipelines to support structured...,Engineered analytics pipelines to support scal...
6,Worked on prompt engineering.,Designed prompt workflows to improve response ...,Used prompt engineering to guide LLM input and...
7,Automated deployments.,Automated deployment workflows to improve rele...,Set up automated deployment workflows to reduc...


In [ ]:
#Evaluation Outputs

for row in results:
    print("INPUT:")
    print(row["input"])
    print("\nEXPECTED:")
    print(row["expected"])
    print("\nPREDICTION:")
    print(row["prediction"])
    print("\n" + "="*100 + "\n")

INPUT:
Built backend for app.

EXPECTED:
Developed backend services for an application, improving scalability, reliability, and support for core user workflows.

PREDICTION:
Developed scalable backend services to support application functionality, improving performance and reducing backend maintenance effort.


INPUT:
Made LLM app for documents.

EXPECTED:
Built an LLM application for document understanding, enabling context-aware responses from retrieved content.

PREDICTION:
Built an LLM application for semantic processing and document generation, improving answer quality and output coherence.


INPUT:
Used AWS to host services.

EXPECTED:
Deployed and managed cloud services on AWS to support scalable, reliable application infrastructure.

PREDICTION:
Deployed serverless workflows on AWS to support event-driven processing and improve backend scalability.


INPUT:
Worked on React pages.

EXPECTED:
Developed React-based user interfaces that improved consistency, responsiveness, and ove

In [ ]:
#Manual Scoring Columns

scored_results = []

for row in results:
    scored_results.append({
        "input": row["input"],
        "expected": row["expected"],
        "prediction": row["prediction"],
        "strength_score": None,
        "clarity_score": None,
        "conciseness_score": None,
        "realism_score": None,
        "notes": ""
    })

score_df = pd.DataFrame(scored_results)
score_df


,input,expected,prediction,strength_score,clarity_score,conciseness_score,realism_score,notes
0,Built backend for app.,"Developed backend services for an application,...",Developed scalable backend services to support...,None,None,None,None,
1,Made LLM app for documents.,Built an LLM application for document understa...,Built an LLM application for semantic processi...,None,None,None,None,
2,Used AWS to host services.,Deployed and managed cloud services on AWS to ...,Deployed serverless workflows on AWS to suppor...,None,None,None,None,
3,Worked on React pages.,Developed React-based user interfaces that imp...,Built responsive React frontend features that ...,None,None,None,None,
4,Connected app with external APIs.,Integrated external APIs into backend workflow...,Integrated third-party APIs and backend servic...,None,None,None,None,
5,Built pipelines for analytics.,Developed data pipelines to support structured...,Engineered analytics pipelines to support scal...,None,None,None,None,
6,Worked on prompt engineering.,Designed prompt workflows to improve response ...,Used prompt engineering to guide LLM input and...,None,None,None,None,
7,Automated deployments.,Automated deployment workflows to improve rele...,Set up automated deployment workflows to reduc...,None,None,None,None,


In [ ]:
#Saving scored results csv

score_df.to_csv("evaluation_results_scored.csv", index=False)
print("Saved evaluation_results_scored.csv")


Saved evaluation_results_scored.csv


In [ ]:
#Downloading scored csv

from google.colab import files
files.download("evaluation_results_scored.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Computing averages after manual scoring
scored = pd.read_csv("evaluation_results_scored_filled.csv")
scored
scored[["strength_score", "clarity_score", "conciseness_score", "realism_score"]].mean()


,0
strength_score,3.625
clarity_score,4.500
conciseness_score,4.000
realism_score,3.750


In [ ]:
#New base model

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Will load unsloth/meta-llama-3.1-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
FastLanguageModel.for_inference(base_model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [ ]:
#Base model Evaluation on eval dataset

base_results = []

for ex in eval_examples:
    prompt = make_prompt(ex)
    prediction = generate_answer(base_model, base_tokenizer, prompt)

    base_results.append({
        "input": ex["input"],
        "expected": ex["output"],
        "base_prediction": prediction,
    })

base_df = pd.DataFrame(base_results)
base_df

,input,expected,base_prediction
0,Built backend for app.,"Developed backend services for an application,...","Built backend for app using Node.js, Express, ..."
1,Made LLM app for documents.,Built an LLM application for document understa...,Made LLM app for documents.
2,Used AWS to host services.,Deployed and managed cloud services on AWS to ...,"Used AWS to host services, including EC2, S3, ..."
3,Worked on React pages.,Developed React-based user interfaces that imp...,Worked on React pages to improve the user expe...
4,Connected app with external APIs.,Integrated external APIs into backend workflow...,"Connected app with external APIs, including a ..."
5,Built pipelines for analytics.,Developed data pipelines to support structured...,Built pipelines for data engineering.\n\n### I...
6,Worked on prompt engineering.,Designed prompt workflows to improve response ...,Worked on prompt engineering to improve the qu...
7,Automated deployments.,Automated deployment workflows to improve rele...,Automated deployments of applications and infr...


In [ ]:
#Fine tuned model on eval dataset

ft_results = []

for ex in eval_examples:
    prompt = make_prompt(ex)
    prediction = generate_answer(model, tokenizer, prompt)

    ft_results.append({
        "input": ex["input"],
        "expected": ex["output"],
        "finetuned_prediction": prediction,
    })

ft_df = pd.DataFrame(ft_results)
ft_df

,input,expected,finetuned_prediction
0,Built backend for app.,"Developed backend services for an application,...",Developed scalable backend services to support...
1,Made LLM app for documents.,Built an LLM application for document understa...,Built an LLM application for semantic processi...
2,Used AWS to host services.,Deployed and managed cloud services on AWS to ...,Deployed serverless workflows on AWS to suppor...
3,Worked on React pages.,Developed React-based user interfaces that imp...,Built responsive React frontend features that ...
4,Connected app with external APIs.,Integrated external APIs into backend workflow...,Integrated third-party APIs and backend servic...
5,Built pipelines for analytics.,Developed data pipelines to support structured...,Engineered analytics pipelines to support scal...
6,Worked on prompt engineering.,Designed prompt workflows to improve response ...,Used prompt engineering to guide LLM input and...
7,Automated deployments.,Automated deployment workflows to improve rele...,Set up automated deployment workflows to reduc...


In [ ]:
#Comparison Table

comparison_df = base_df.merge(ft_df, on=["input", "expected"])
comparison_df

,input,expected,base_prediction,finetuned_prediction
0,Built backend for app.,"Developed backend services for an application,...","Built backend for app using Node.js, Express, ...",Developed scalable backend services to support...
1,Made LLM app for documents.,Built an LLM application for document understa...,Made LLM app for documents.,Built an LLM application for semantic processi...
2,Used AWS to host services.,Deployed and managed cloud services on AWS to ...,"Used AWS to host services, including EC2, S3, ...",Deployed serverless workflows on AWS to suppor...
3,Worked on React pages.,Developed React-based user interfaces that imp...,Worked on React pages to improve the user expe...,Built responsive React frontend features that ...
4,Connected app with external APIs.,Integrated external APIs into backend workflow...,"Connected app with external APIs, including a ...",Integrated third-party APIs and backend servic...
5,Built pipelines for analytics.,Developed data pipelines to support structured...,Built pipelines for data engineering.\n\n### I...,Engineered analytics pipelines to support scal...
6,Worked on prompt engineering.,Designed prompt workflows to improve response ...,Worked on prompt engineering to improve the qu...,Used prompt engineering to guide LLM input and...
7,Automated deployments.,Automated deployment workflows to improve rele...,Automated deployments of applications and infr...,Set up automated deployment workflows to reduc...


In [ ]:
#Saving comparison results

comparison_df.to_csv("base_vs_finetuned_comparison.csv", index=False)
print("Saved base_vs_finetuned_comparison.csv")

Saved base_vs_finetuned_comparison.csv


In [ ]:
#Downloading the comparison csv

from google.colab import files
files.download("base_vs_finetuned_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Importing manual rated comparison results csv

scored_comparision=pd.read_csv("base_vs_finetuned_comparison_rated.csv")
scored_comparision

,input,expected,base_prediction,finetuned_prediction,base_strength,base_clarity,base_conciseness,base_realism,ft_strength,ft_clarity,ft_conciseness,ft_realism,notes
0,Built backend for app.,"Developed backend services for an application,...","Built backend for app using Node.js, Express, ...",Developed scalable backend services to support...,3.0,2.0,1.0,2.0,4.0,5.0,4.0,4.0,Base adds unsupported tech stack details and e...
1,Made LLM app for documents.,Built an LLM application for document understa...,Made LLM app for documents.,Built an LLM application for semantic processi...,1.0,3.0,5.0,5.0,3.0,4.0,4.0,2.0,Base mostly repeats the original bullet. Fine-...
2,Used AWS to host services.,Deployed and managed cloud services on AWS to ...,"Used AWS to host services, including EC2, S3, ...",Deployed serverless workflows on AWS to suppor...,2.0,4.0,4.0,2.0,3.0,4.0,4.0,2.0,Both versions invent specifics. Base adds EC2/...
3,Worked on React pages.,Developed React-based user interfaces that imp...,Worked on React pages to improve the user expe...,Built responsive React frontend features that ...,2.0,2.0,1.0,4.0,4.0,5.0,4.0,4.0,Base has explanation leakage and only a small ...
4,Connected app with external APIs.,Integrated external APIs into backend workflow...,"Connected app with external APIs, including a ...",Integrated third-party APIs and backend servic...,3.0,2.0,1.0,2.0,4.0,5.0,4.0,5.0,Base injects specific API types plus explanati...
5,Built pipelines for analytics.,Developed data pipelines to support structured...,Built pipelines for data engineering.\n\n### I...,Engineered analytics pipelines to support scal...,1.0,1.0,1.0,2.0,4.0,4.0,4.0,4.0,Base output breaks badly with repetition and p...
6,Worked on prompt engineering.,Designed prompt workflows to improve response ...,Worked on prompt engineering to improve the qu...,Used prompt engineering to guide LLM input and...,2.0,2.0,1.0,4.0,3.0,4.0,4.0,4.0,Base includes explanation text; fine-tuned is ...
7,Automated deployments.,Automated deployment workflows to improve rele...,Automated deployments of applications and infr...,Set up automated deployment workflows to reduc...,3.0,2.0,1.0,5.0,4.0,5.0,4.0,5.0,Base is reasonable but hurt by explanation lea...


In [ ]:
#Computing comparison averages

base_cols = [
    "base_strength",
    "base_clarity",
    "base_conciseness",
    "base_realism"
]

ft_cols = [
    "ft_strength",
    "ft_clarity",
    "ft_conciseness",
    "ft_realism"
]

print("Base Model Averages")
print(scored_comparision[base_cols].mean())

print("\nFine-Tuned Model Averages")
print(scored_comparision[ft_cols].mean())

Base Model Averages
base_strength       2.125
base_clarity        2.250
base_conciseness    1.875
base_realism        3.250
dtype: float64

Fine-Tuned Model Averages
ft_strength       3.625
ft_clarity        4.500
ft_conciseness    4.000
ft_realism        3.750
dtype: float64


In [ ]:
#Improvement observed

delta = pd.Series({
    "strength": scored_comparision["ft_strength"].mean() - scored_comparision["base_strength"].mean(),
    "clarity": scored_comparision["ft_clarity"].mean() - scored_comparision["base_clarity"].mean(),
    "conciseness": scored_comparision["ft_conciseness"].mean() - scored_comparision["base_conciseness"].mean(),
    "realism": scored_comparision["ft_realism"].mean() - scored_comparision["base_realism"].mean(),
})

print("Improvement (Fine-tuned minus Base)")
print(delta)

Improvement (Fine-tuned minus Base)
strength       1.500
clarity        2.250
conciseness    2.125
realism        0.500
dtype: float64


In [ ]:
#Identifying failure Patterns

scored_comparision[[
    "input",
    "base_prediction",
    "finetuned_prediction"
]]

,input,base_prediction,finetuned_prediction
0,Built backend for app.,"Built backend for app using Node.js, Express, ...",Developed scalable backend services to support...
1,Made LLM app for documents.,Made LLM app for documents.,Built an LLM application for semantic processi...
2,Used AWS to host services.,"Used AWS to host services, including EC2, S3, ...",Deployed serverless workflows on AWS to suppor...
3,Worked on React pages.,Worked on React pages to improve the user expe...,Built responsive React frontend features that ...
4,Connected app with external APIs.,"Connected app with external APIs, including a ...",Integrated third-party APIs and backend servic...
5,Built pipelines for analytics.,Built pipelines for data engineering.\n\n### I...,Engineered analytics pipelines to support scal...
6,Worked on prompt engineering.,Worked on prompt engineering to improve the qu...,Used prompt engineering to guide LLM input and...
7,Automated deployments.,Automated deployments of applications and infr...,Set up automated deployment workflows to reduc...


In [ ]:
#The end of Common ML loop